# Sea-surface salinity and potential temperature restoring

### Regrid surface salinity and potential temperature from land filled WOA dataset to the ocean model grid.

In [1]:
import xarray as xr
import numpy as np
from datetime import datetime
import os, subprocess
import xesmf

### Read tx1_12 grid

In [2]:
ds_out = xr.open_dataset('../topography/ocean_topog_tx1_12v1_260213.nc').rename({'x': 'lon','y': 'lat'})

### Read WOA file with land fill

In [3]:
# WOA sfc state file with land fill, created using create_filled_sfc.py
fname = '/glade/campaign/cgd/oce/datasets/cesm/WOA_MOM6/woa18_sfc_state_filled.nc'
woa = xr.open_dataset(fname, decode_times=False)

In [4]:
# average between two-layers (depth = 0 and depth = 10, depth indices 0 and 2)
woa_s_an_surface_ave = woa.s_an.isel(depth=[0,2]).mean('depth')
woa_theta0_surface_ave = woa.theta0.isel(depth=[0,2]).mean('depth')

In [5]:
def regrid_tracer(fld, ds_in, ds_out, method='bilinear'):

    regrid = xesmf.Regridder(
        ds_in,
        ds_out,
        method=method,
        periodic=True,
    )
    fld_out = regrid(fld)
    return fld_out

In [6]:
ds_out_s_an = regrid_tracer(woa_s_an_surface_ave, woa_s_an_surface_ave, ds_out)

In [7]:
ds_out_theta0 = regrid_tracer(woa_theta0_surface_ave, woa_theta0_surface_ave, ds_out)

### Create state file for MOM6
We opted to create this file via ncgen to avoid issues with FMS reading the netCDF file.

In [8]:
!ncgen -o state_restore_tx1_12.nc state_restore_tx1_12.cdl

In [9]:
state = xr.open_dataset('state_restore_tx1_12.nc', decode_times=False)

In [10]:
ds_out

<xarray.Dataset> Size: 336MB
Dimensions:  (ny: 3240, nx: 4320)
Dimensions without coordinates: ny, nx
Data variables:
    lat      (ny, nx) float64 112MB -80.06 -80.06 -80.06 ... 50.0 50.0 50.0
    lon      (ny, nx) float64 112MB -287.0 -286.9 -286.8 ... 73.0 73.0 73.0
    mask     (ny, nx) int32 56MB 0 0 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0 0 0 0
    depth    (ny, nx) float32 56MB ...
Attributes:
    date_created:  2026-02-20T13:18:02.079072
    title:         MOM6 topography file
    min_depth:     9.5
    max_depth:     10419.4609375

In [11]:
jm, im = ds_out.lat.shape
state['LAT'] = ds_out.lat[:,int(im/2)].values
state['LON'] = ds_out.lon[int(jm/2),:].values

Overwrite salt and thetao

In [12]:
state['salt'].values[:] = ds_out_s_an.values[:]

In [13]:
state['theta0'].values[:] = ds_out_theta0.values[:]

TODO: Compare original and remapped fields

In [14]:
# Global attrs
state.attrs['title'] = 'surface salinity and potential temperature from WOA filled over continents'
state.attrs['src_file'] = fname
state.attrs['dst_grid_name'] = 'tx1_12'
state.attrs['author'] = 'Gustavo Marques (gmarques@ucar.edu)'
state.attrs['date'] = datetime.now().isoformat()
state.attrs['created_using'] = 'https://github.com/NCAR/tx1_12/state_restoring/state_restoring.ipynb'
# save 
fname1 = 'state_restore_tx1_12_{}{}{}.nc'.format(datetime.now().isoformat()[0:4],datetime.now().isoformat()[5:7],
       datetime.now().isoformat()[8:10])
state.to_netcdf(fname1)

/glade/u/apps/opt/conda/envs/npl-2024b/lib/python3.11/site-packages/dask/config.py:789: FutureWarning: Dask configuration key 'allowed-failures' has been deprecated; please use 'distributed.scheduler.allowed-failures' instead
  warnings.warn(
